In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
import numpy as np
import cv2
import os
import h5py
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import random

In [2]:
# Constants (define these at the top)
IMAGE_SIZE = (256, 256)
DATASET_DIR = "/kaggle/input/figshare-brain-tumor-dataset/dataset/data"  # Replace with your actual dataset path


In [3]:
def load_mat_files(dataset_dir):
    images, masks = [], []
    files = sorted(os.listdir(dataset_dir))
    
    for file in files:
        if file.endswith(".mat"):
            file_path = os.path.join(dataset_dir, file)
            with h5py.File(file_path, 'r') as data:
                try:
                    img = np.array(data['cjdata']['image'], dtype=np.float32)
                    mask = np.array(data['cjdata']['tumorMask'], dtype=np.uint8)
                    
                    img = cv2.resize(img, IMAGE_SIZE) / 255.0  # Normalize image
                    mask = cv2.resize(mask, IMAGE_SIZE)
                    mask = (mask > 0).astype(np.uint8)  # Ensure binary mask
                    
                    images.append(img[..., np.newaxis])  # Add channel dimension
                    masks.append(mask[..., np.newaxis])
                except KeyError as e:
                    print(f"Skipping {file}: KeyError - {e}")
    return np.array(images), np.array(masks)

In [4]:
# Data Augmentation function
def augment_data(images, masks):
    augmented_images = []
    augmented_masks = []
    
    for img, mask in zip(images, masks):
        # Original
        augmented_images.append(img)
        augmented_masks.append(mask)
        
        # Horizontal flip
        flip_img = np.flip(img, axis=1)
        flip_mask = np.flip(mask, axis=1)
        augmented_images.append(flip_img)
        augmented_masks.append(flip_mask)
        # Rotation 90 degrees
        rot_img = np.rot90(img, k=1, axes=(0, 1))
        rot_mask = np.rot90(mask, k=1, axes=(0, 1))
        augmented_images.append(rot_img)
        augmented_masks.append(rot_mask)
        
        # Brightness variation (random)
        brightness = np.random.uniform(0.8, 1.2)
        bright_img = np.clip(img * brightness, 0, 1)
        augmented_images.append(bright_img)
        augmented_masks.append(mask)
    
    return np.array(augmented_images), np.array(augmented_masks)

In [5]:
# Improved U-Net Model with Dropout and Batch Normalization
def improved_unet_model(input_shape=(256, 256, 1)):
    inputs = layers.Input(input_shape)
    
    # Encoder
    c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(inputs)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c1)
    c1 = layers.BatchNormalization()(c1)
    p1 = layers.MaxPooling2D((2,2))(c1)
    p1 = layers.Dropout(0.1)(p1)
    
    c2 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c2)
    c2 = layers.BatchNormalization()(c2)
    p2 = layers.MaxPooling2D((2,2))(c2)
    p2 = layers.Dropout(0.2)(p2)
    
    c3 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(p2)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(c3)
    c3 = layers.BatchNormalization()(c3)
    p3 = layers.MaxPooling2D((2,2))(c3)
    p3 = layers.Dropout(0.3)(p3)
    
    c4 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(p3)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(c4)
    c4 = layers.BatchNormalization()(c4)
    p4 = layers.MaxPooling2D((2,2))(c4)
    p4 = layers.Dropout(0.4)(p4)
    
    # Bottleneck
    c5 = layers.Conv2D(1024, (3,3), activation='relu', padding='same')(p4)
    c5 = layers.BatchNormalization()(c5)
    c5 = layers.Conv2D(1024, (3,3), activation='relu', padding='same')(c5)
    c5 = layers.BatchNormalization()(c5)
    c5 = layers.Dropout(0.5)(c5)
    
    # Decoder
    u6 = layers.Conv2DTranspose(512, (2,2), strides=(2,2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(u6)
    c6 = layers.BatchNormalization()(c6)
    c6 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(c6)
    c6 = layers.BatchNormalization()(c6)
    c6 = layers.Dropout(0.4)(c6)
    
    u7 = layers.Conv2DTranspose(256, (2,2), strides=(2,2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(u7)
    c7 = layers.BatchNormalization()(c7)
    c7 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(c7)
    c7 = layers.BatchNormalization()(c7)
    c7 = layers.Dropout(0.3)(c7)
    
    u8 = layers.Conv2DTranspose(128, (2,2), strides=(2,2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(u8)
    c8 = layers.BatchNormalization()(c8)
    c8 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c8)
    c8 = layers.BatchNormalization()(c8)
    c8 = layers.Dropout(0.2)(c8)
    
    u9 = layers.Conv2DTranspose(64, (2,2), strides=(2,2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(u9)
    c9 = layers.BatchNormalization()(c9)
    c9 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c9)
    c9 = layers.BatchNormalization()(c9)
    c9 = layers.Dropout(0.1)(c9)
    
    outputs = layers.Conv2D(1, (1,1), activation='sigmoid')(c9)
    
    return Model(inputs, outputs)


In [6]:
# Custom Dice Loss
def dice_loss(y_true, y_pred):
    smooth = 1e-6
    # Convert to float32 to ensure consistent data types
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    dice = (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)
    return 1 - dice


In [7]:
# Combined loss function
def bce_dice_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)


In [8]:
# Dice coefficient metric
def dice_coefficient(y_true, y_pred):
    smooth = 1e-6
    # Convert to float32 to ensure consistent data types
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)


In [9]:
# IoU metric
def iou_score(y_true, y_pred):
    smooth = 1e-6
    # Convert to float32 to ensure consistent data types
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

In [10]:
# Load and process data
print("Loading dataset...")
images, masks = load_mat_files(DATASET_DIR)
print(f"Original dataset: {len(images)} images, {len(masks)} masks")


Loading dataset...
Original dataset: 3064 images, 3064 masks


In [11]:
# Augment training data
print("Augmenting data...")
augmented_images, augmented_masks = augment_data(images, masks)
print(f"Augmented dataset: {len(augmented_images)} images, {len(augmented_masks)} masks")


Augmenting data...
Augmented dataset: 12256 images, 12256 masks


In [12]:
# Split into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(augmented_images, augmented_masks, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=1/3, random_state=42)
print(f"Train: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")


Train: 8579, Validation: 2451, Test: 1226


In [ ]:
# Create and compile model
model = improved_unet_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=bce_dice_loss,
    metrics=['accuracy', dice_coefficient, iou_score]
)

# Callbacks
early_stopping = callbacks.EarlyStopping(
    monitor='val_dice_coefficient',
    mode='max',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_dice_coefficient',
    mode='max',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

checkpoint = callbacks.ModelCheckpoint(
    "best_brain_tumor_model.keras",
    monitor='val_dice_coefficient',
    mode='max',
    save_best_only=True,
    verbose=1
)

# TensorBoard callback for visualization
tensorboard_callback = callbacks.TensorBoard(
    log_dir="./logs",
    histogram_freq=1,
    update_freq='epoch'
)

# Train model
print("Training model...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping, lr_scheduler, checkpoint, tensorboard_callback]
)

Training model...
Epoch 1/100
537/537 ━━━━━━━━━━━━━━━━━━━━ 0s 406ms/step - accuracy: 0.7070 - dice_coefficient: 0.0410 - iou_score: 0.0210 - loss: 1.6132
Epoch 1: val_dice_coefficient improved from -inf to 0.07109, saving model to best_brain_tumor_model.keras
537/537 ━━━━━━━━━━━━━━━━━━━━ 313s 486ms/step - accuracy: 0.7072 - dice_coefficient: 0.0410 - iou_score: 0.0210 - loss: 1.6130 - val_accuracy: 0.9244 - val_dice_coefficient: 0.0711 - val_iou_score: 0.0369 - val_loss: 1.3816 - learning_rate: 1.0000e-04
Epoch 2/100
537/537 ━━━━━━━━━━━━━━━━━━━━ 0s 373ms/step - accuracy: 0.9733 - dice_coefficient: 0.0863 - iou_score: 0.0453 - loss: 1.2066
Epoch 2: val_dice_coefficient improved from 0.07109 to 0.16685, saving model to best_brain_tumor_model.keras
537/537 ━━━━━━━━━━━━━━━━━━━━ 224s 417ms/step - accuracy: 0.9733 - dice_coefficient: 0.0863 - iou_score: 0.0454 - loss: 1.2065 - val_accuracy: 0.9485 - val_dice_coefficient: 0.1668 - val_iou_score: 0.0914 - val_loss: 1.0188 - learning_rate: 1.00

In [ ]:
# Save final model
model.save("brain_tumor_unet_final.keras")


In [ ]:
# Visualize training history
def plot_training_history(history):
    # Plot loss
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot accuracy
    plt.subplot(1, 3, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    # Plot Dice coefficient
    plt.subplot(1, 3, 3)
    plt.plot(history.history['dice_coefficient'], label='Training Dice')
    plt.plot(history.history['val_dice_coefficient'], label='Validation Dice')
    plt.title('Dice Coefficient')
    plt.xlabel('Epoch')
    plt.ylabel('Dice')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.show()

In [ ]:
print("Plotting training history...")
plot_training_history(history)


In [ ]:
# Model evaluation
print("Evaluating model...")
evaluation = model.evaluate(X_test, y_test)
print(f"Test Loss: {evaluation[0]}")
print(f"Test Accuracy: {evaluation[1]}")
print(f"Test Dice Coefficient: {evaluation[2]}")
print(f"Test IoU Score: {evaluation[3]}")


In [ ]:
# Generate predictions
y_pred = model.predict(X_test)
y_pred_bin = (y_pred > 0.5).astype(np.uint8)
y_test_bin = (y_test > 0.5).astype(np.uint8)

In [ ]:
# Calculate metrics
dice_score = np.mean([dice_coefficient(y_true, y_pred) for y_true, y_pred in zip(y_test_bin, y_pred_bin)])
iou = np.mean([iou_score(y_true, y_pred) for y_true, y_pred in zip(y_test_bin, y_pred_bin)])

print(f"Dice Score: {dice_score}")
print(f"IoU Score: {iou}")


In [ ]:
# Confusion matrix
flat_y_test = y_test_bin.flatten()
flat_y_pred = y_pred_bin.flatten()
cm = confusion_matrix(flat_y_test, flat_y_pred)

tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)
precision = tp / (tp + fp)
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("Confusion Matrix:")
print(cm)
print(f"Accuracy: {accuracy:.4f}")
print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Precision: {precision:.4f}")

In [ ]:
# Function to visualize random predictions
def visualize_random_predictions(X_test, y_test, y_pred, num_samples=5):
    indices = random.sample(range(len(X_test)), num_samples)
    
    plt.figure(figsize=(15, 4*num_samples))
    
    for i, idx in enumerate(indices):
        # Original image
        plt.subplot(num_samples, 3, i*3 + 1)
        plt.imshow(X_test[idx].squeeze(), cmap='gray')
        plt.title(f"Original Image {idx}")
        plt.axis('off')
        
        # Ground truth mask
        plt.subplot(num_samples, 3, i*3 + 2)
        plt.imshow(y_test[idx].squeeze(), cmap='viridis')
        plt.title(f"Ground Truth Mask {idx}")
        plt.axis('off')
        
        # Predicted mask
        plt.subplot(num_samples, 3, i*3 + 3)
        plt.imshow(y_pred[idx].squeeze(), cmap='viridis')
        plt.title(f"Predicted Mask {idx}\nDice: {dice_coefficient(y_test[idx], y_pred_bin[idx]):.4f}")
        plt.axis('off')
    
    plt.tight_layout()
    plt.savefig('random_predictions.png')
    plt.show()

In [ ]:
# Visualize predictions on random test samples
print("Visualizing random predictions...")
visualize_random_predictions(X_test, y_test, y_pred)

In [ ]:
# Function to visualize comparison across image slices
def visualize_3d_comparison(X_test, y_test, y_pred_bin, index=None):
    if index is None:
        # Find an example with a good size tumor
        tumor_sizes = [np.sum(y_test[i]) for i in range(len(y_test))]
        index = np.argsort(tumor_sizes)[-5]  # Get one of the larger tumors
    
    img = X_test[index].squeeze()
    true_mask = y_test[index].squeeze()
    pred_mask = y_pred_bin[index].squeeze()
    
    # Create colormap figures
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title('Original MRI')
    axes[0].axis('off')
    
    # True mask
    axes[1].imshow(img, cmap='gray')
    axes[1].imshow(true_mask, cmap='hot', alpha=0.4)
    axes[1].set_title('Ground Truth Mask')
    axes[1].axis('off')
    
    # Predicted mask
    axes[2].imshow(img, cmap='gray')
    axes[2].imshow(pred_mask, cmap='hot', alpha=0.4)
    dice = dice_coefficient(y_test[index], y_pred_bin[index])
    axes[2].set_title(f'Predicted Mask (Dice: {dice:.4f})')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig('3d_visualization.png')
    plt.show()


In [ ]:
# Visualize 3D comparison
print("Visualizing 3D comparison...")
visualize_3d_comparison(X_test, y_test, y_pred_bin)

In [ ]:
# Function to create interactive visualization tool
def create_interactive_visualization():
    """
    Creates a function that can be called in a Jupyter notebook
    to interactively visualize predictions on test data
    """
    import ipywidgets as widgets
    from IPython.display import display
    
    def view_sample(index):
        img = X_test[index].squeeze()
        true_mask = y_test[index].squeeze()
        pred_mask = y_pred[index].squeeze() > 0.5
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        axes[0].imshow(img, cmap='gray')
        axes[0].set_title('Original MRI')
        axes[0].axis('off')
        
        axes[1].imshow(img, cmap='gray')
        axes[1].imshow(true_mask, cmap='hot', alpha=0.5)
        axes[1].set_title('Ground Truth Mask')
        axes[1].axis('off')
        
        axes[2].imshow(img, cmap='gray')
        axes[2].imshow(pred_mask, cmap='hot', alpha=0.5)
        dice = dice_coefficient(y_test[index], y_pred_bin[index])
        axes[2].set_title(f'Predicted Mask (Dice: {dice:.4f})')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    slider = widgets.IntSlider(min=0, max=len(X_test)-1, step=1, value=0, description='Sample:')
    widgets.interact(view_sample, index=slider)

In [ ]:
print("Done! You can now use the improved model for brain tumor segmentation.")
print("The model has been saved as 'brain_tumor_unet_final.keras' and 'best_brain_tumor_model.keras'.")
print("Visualizations have been saved as PNG files.")